In [ ]:
import stackview
from tomobase.phantoms import get_nanocage
from tomobase import processes, tiltschemes

volume = get_nanocage()

angles = tiltschemes.GRS().generate_angles(70)
sinogram = processes.project(volume, angles)

sinogram2 = processes.poisson_noise(sinogram)
sinogram3 = processes.gaussian_filter(sinogram)


recon = processes.astra_reconstruct(sinogram)
recon2 = processes.optomo_reconstruct(sinogram2)

result = processes.ssim(volume, recon)
processes(volume, recon, results=[result])

result2 = processes.ssim(sinogram, sinogram2, axial=0)
processes.ssim(sinogram, sinogram3, results=[result2], axial=0)

# Just a look internally 
@process_hook(name='Structural Similarity',category=categories['Quality Metrics'], output_structure =[AxisSpec('ssim', 'a.u.')])
def ssim( image:BaseImageModel, reference:BaseImageModel):
    xp =image.data.__array_namespace__()
    if image.data.dtype == xp.uint8:
        data_range = 255
    elif image.data.dtype == xp.float32 or image.data.dtype == xp.float64:
        data_range = 1.0
        if image.data.max() > 1.0:
            data_range = image.data.max() - image.data.min()
    value = proxy.skimage.metrics.structural_similarity(image.data, reference.data, data_range=data_range)
    return value
Ideally the process hook will create a AnalsisData Class with the data being an xarray for each of the output variable
